In [4]:
"""
PostgreSQL SQ8 Benchmark - 's Approach
Each SQ8 dimension stored as a separate column

Schema:
  id | x1 | x2 | x3 | ... | x1024
  ---+----+----+----+-----+------
  1  | 127| 43 | 198| ... | 89

Retrieval SQL ( exact formula):
  SELECT id,
    (CAST(x1 AS INT) * :q1) +
    (CAST(x2 AS INT) * :q2) + ...
    (CAST(x1024 AS INT) * :q1024) AS dot
  FROM vectors
  ORDER BY dot DESC
  LIMIT 100;

Prerequisites:
  - PostgreSQL running (same as before)
  - NO pgvector extension needed! Pure SQL approach
"""

import pickle
import time
import json
import numpy as np
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
import os

# ========================
# CONFIGURATION
# ========================
QRELS_PATH = "/Users/vikashpr/Dev/Python/FinanceRAG/icaif-24-finance-rag-challenge/FinDER_qrels.tsv"
POSTGRES_CONNECTION = "postgresql://vikashpr@localhost:5432/financerag"

# SQ8 quantized embeddings
CORPUS_EMBEDDINGS_SQ8_PATH = "./encoded_data/corpus_embeddings_sq8.pkl"
QUERY_EMBEDDINGS_SQ8_PATH  = "./encoded_data/query_embeddings_sq8.pkl"

DIM = 1024  # embedding dimension

TABLE_NAME = "finder_sq8_columns"  # table with 1024 INT columns

# ========================
# STEP 1 - LOAD SQ8 DATA
# ========================

def load_sq8_embeddings():
    """
    Load SQ8 quantized embeddings (uint8).
    We do NOT dequantize - we store raw uint8 values as integers in DB.
    The query vector stays float32, per 's SQL: CAST(x AS INT) * q_float
    """
    print("\n📥 Loading SQ8 quantized embeddings...")

    with open(CORPUS_EMBEDDINGS_SQ8_PATH, 'rb') as f:
        corpus_data = pickle.load(f)

    with open(QUERY_EMBEDDINGS_SQ8_PATH, 'rb') as f:
        query_data = pickle.load(f)

    corpus_emb = corpus_data['embeddings']  # uint8, shape (N, 1024)
    query_emb  = query_data['embeddings']   # uint8, shape (Q, 1024)
    corpus_ids = corpus_data['ids']
    query_ids  = query_data['ids']
    quantizer_params = corpus_data['quantizer_params']

    print(f"✅ Corpus: {corpus_emb.shape}, dtype={corpus_emb.dtype}")
    print(f"✅ Queries: {query_emb.shape}, dtype={query_emb.dtype}")
    print(f"   Quantizer scale={quantizer_params['scale']:.6f}, offset={quantizer_params['offset']:.6f}")

    return corpus_emb, corpus_ids, query_emb, query_ids, quantizer_params

def load_qrels():
    """Load ground truth relevance judgments"""
    df = pd.read_csv(QRELS_PATH, sep="\t")
    qrels = df.groupby("query_id").apply(
        lambda g: dict(zip(g["corpus_id"], g["score"]))
    ).to_dict()
    print(f"✅ Loaded {len(qrels)} query judgments")
    return qrels

# ========================
# STEP 2 - CREATE TABLE
# ========================

def create_table(conn):
    """
    Create table with 1024 individual INT columns (one per SQ8 dimension).

    Schema:
      id      SERIAL PRIMARY KEY
      doc_id  TEXT
      x1      SMALLINT   (range 0-255 fits in SMALLINT = 2 bytes)
      x2      SMALLINT
      ...
      x1024   SMALLINT
    """
    print(f"\n📦 Creating table: {TABLE_NAME}")
    print(f"   Schema: id, doc_id, x1, x2, ..., x{DIM}  ({DIM+2} columns total)")

    cur = conn.cursor()

    # Drop existing
    cur.execute(f"DROP TABLE IF EXISTS {TABLE_NAME};")

    # Build column definitions  (x1 to x1024, SMALLINT for uint8 values 0-255)
    col_defs = ", ".join(f"x{i+1} SMALLINT" for i in range(DIM))

    cur.execute(f"""
        CREATE TABLE {TABLE_NAME} (
            id     SERIAL PRIMARY KEY,
            doc_id TEXT NOT NULL,
            {col_defs}
        );
    """)

    conn.commit()
    cur.close()
    print(f"✅ Table created with {DIM + 2} columns")

# ========================
# STEP 3 - INSERT CORPUS
# ========================

def insert_corpus(conn, corpus_emb, corpus_ids, batch_size=200):
    """
    Insert all corpus documents.
    Each row = one document with 1024 integer columns.
    """
    print(f"\n📤 Inserting {len(corpus_ids)} documents ({DIM} columns each)...")

    cur = conn.cursor()

    # Column names: doc_id, x1, x2, ..., x1024
    col_names = "doc_id, " + ", ".join(f"x{i+1}" for i in range(DIM))
    insert_sql = f"INSERT INTO {TABLE_NAME} ({col_names}) VALUES %s"

    inserted = 0
    for start in range(0, len(corpus_ids), batch_size):
        end = min(start + batch_size, len(corpus_ids))

        batch_rows = []
        for idx in range(start, end):
            row = [corpus_ids[idx]] + corpus_emb[idx].tolist()  # doc_id + 1024 ints
            batch_rows.append(tuple(row))

        execute_values(cur, insert_sql, batch_rows)
        inserted += (end - start)

        if inserted % 2000 == 0 or inserted == len(corpus_ids):
            print(f"   Inserted {inserted}/{len(corpus_ids)} documents")

    conn.commit()
    cur.close()
    print(f"✅ All {len(corpus_ids)} documents inserted")

# ========================
# STEP 4 - BUILD INDEX
# ========================

def build_index(conn, index_type="NONE"):
    """
    Optional: Create a btree index on doc_id for lookups.
    No vector index is used - retrieval is pure SQL computation.
    """
    print(f"\n🔨 Index setup: {index_type}")
    cur = conn.cursor()

    if index_type == "NONE":
        print("   No vector index - pure sequential SQL scan")
        build_time = 0.0

    elif index_type == "DOCID_INDEX":
        # Just a btree index on doc_id column (helps lookups, not retrieval)
        start = time.time()
        cur.execute(f"CREATE INDEX idx_doc_id ON {TABLE_NAME}(doc_id);")
        conn.commit()
        build_time = time.time() - start
        print(f"   Created btree index on doc_id in {build_time:.2f}s")

    cur.close()
    print(f"✅ Index setup complete in {build_time:.2f}s")
    return build_time

# ========================
# STEP 5 - RETRIEVAL SQL
# ========================

def build_dot_product_sql(query_vector_float, top_k=10):
    """
    Build 's SQL:

      SELECT id, doc_id,
        (CAST(x1 AS INT) * :q1) +
        (CAST(x2 AS INT) * :q2) + ...
        (CAST(x1024 AS INT) * :q1024) AS dot
      FROM vectors
      ORDER BY dot DESC
      LIMIT 100;

    Note: query_vector_float is the dequantized float32 query.
    The corpus stays as uint8 integers in the DB.
    The SQL casts each DB integer and multiplies by the float query value.
    This computes the approximate inner product (dot product).
    """
    # Build dot product expression
    dot_terms = " + ".join(
        f"(CAST(x{i+1} AS INT) * {float(query_vector_float[i]):.8f})"
        for i in range(DIM)
    )

    sql = f"""
        SELECT doc_id,
               {dot_terms} AS dot
        FROM   {TABLE_NAME}
        ORDER  BY dot DESC
        LIMIT  {top_k};
    """
    return sql

def dequantize_query(query_uint8, quantizer_params):
    """
    Dequantize query vector: uint8 → float32.
    This is the q_i values multiplied against the stored SQ8 integers.
    """
    scale  = quantizer_params['scale']
    offset = quantizer_params['offset']
    q_float = query_uint8.astype(np.float32) * scale + offset
    return q_float

def search_queries(conn, query_emb, query_ids, quantizer_params, top_k=10):
    """
    Run 's SQL retrieval for all queries.

    For each query:
      1. Dequantize query vector (uint8 → float32)
      2. Build SQL with float query values
      3. Execute against the INT columns in the DB
      4. Rank by dot product score
    """
    print(f"\n🔍 Searching {len(query_ids)} queries using 's SQL...")
    print(f"   Method: Pure SQL dot product (CAST(x AS INT) * q_float)")
    print(f"   Table: {TABLE_NAME} ({DIM} integer columns)")

    cur = conn.cursor()

    results     = {}
    query_times = []

    for i, (query_id, query_uint8) in enumerate(zip(query_ids, query_emb)):

        # Dequantize query to float32
        q_float = dequantize_query(query_uint8, quantizer_params)

        # Build and run 's SQL
        sql = build_dot_product_sql(q_float, top_k=top_k)

        t_start = time.time()
        cur.execute(sql)
        rows = cur.fetchall()
        query_time = time.time() - t_start
        query_times.append(query_time)

        # Store results: {doc_id: dot_score}
        results[query_id] = {}
        for doc_id, dot in rows:
            results[query_id][doc_id] = float(dot)

        if (i + 1) % 20 == 0:
            avg_ms = np.mean(query_times[-20:]) * 1000
            print(f"   Processed {i+1}/{len(query_ids)} queries (avg: {avg_ms:.1f}ms/query)")

    cur.close()

    total_time = sum(query_times)
    avg_time   = np.mean(query_times)

    print(f"\n✅ Search complete:")
    print(f"   Total time: {total_time:.2f}s")
    print(f"   Avg per query: {avg_time*1000:.2f}ms")

    return results, total_time, avg_time

# ========================
# METRICS
# ========================

def calculate_ndcg(qrels, results, k):
    scores = []
    for qid, relevant in qrels.items():
        if qid not in results:
            continue
        ranked = sorted(results[qid].items(), key=lambda x: x[1], reverse=True)[:k]
        dcg  = sum(relevant.get(d, 0) / np.log2(i+2) for i,(d,_) in enumerate(ranked))
        idcg = sum(v / np.log2(i+2) for i,v in enumerate(sorted(relevant.values(), reverse=True)[:k]))
        scores.append(dcg/idcg if idcg > 0 else 0)
    return float(np.mean(scores))

def calculate_recall(qrels, results, k):
    scores = []
    for qid, relevant in qrels.items():
        if qid not in results:
            continue
        retrieved = set(list(results[qid].keys())[:k])
        rel_set   = set(relevant.keys())
        if rel_set:
            scores.append(len(retrieved & rel_set) / len(rel_set))
    return float(np.mean(scores))

def calculate_precision(qrels, results, k):
    scores = []
    for qid, relevant in qrels.items():
        if qid not in results:
            continue
        retrieved = list(results[qid].keys())[:k]
        rel_set   = set(relevant.keys())
        if retrieved:
            scores.append(len([d for d in retrieved if d in rel_set]) / len(retrieved))
    return float(np.mean(scores))

def calculate_map(qrels, results, k):
    scores = []
    for qid, relevant in qrels.items():
        if qid not in results:
            continue
        retrieved  = list(results[qid].keys())[:k]
        rel_set    = set(relevant.keys())
        if not rel_set:
            continue
        n_rel, psum = 0, 0.0
        for i, d in enumerate(retrieved):
            if d in rel_set:
                n_rel += 1
                psum  += n_rel / (i+1)
        scores.append(psum / len(rel_set))
    return float(np.mean(scores))

def evaluate(qrels, results):
    print("\n📈 Evaluating metrics...")
    return {
        "ndcg@1":       calculate_ndcg(qrels, results, 1),
        "ndcg@5":       calculate_ndcg(qrels, results, 5),
        "ndcg@10":      calculate_ndcg(qrels, results, 10),
        "recall@1":     calculate_recall(qrels, results, 1),
        "recall@5":     calculate_recall(qrels, results, 5),
        "recall@10":    calculate_recall(qrels, results, 10),
        "precision@1":  calculate_precision(qrels, results, 1),
        "precision@5":  calculate_precision(qrels, results, 5),
        "precision@10": calculate_precision(qrels, results, 10),
        "map@10":       calculate_map(qrels, results, 10),
    }

# ========================
# MAIN
# ========================

def main():
    print("="*60)
    print("POSTGRESQL SQ8 BENCHMARK ('s Column-per-Dimension Approach)")
    print("="*60)
    print(f"\n📐 Schema: 1 row per document, {DIM} integer columns (x1…x{DIM})")
    print(f"🔍 Retrieval: Pure SQL dot product - no pgvector needed!")

    # Load data
    corpus_emb, corpus_ids, query_emb, query_ids, quantizer_params = load_sq8_embeddings()
    qrels = load_qrels()

    # Connect to PostgreSQL
    print(f"\n🔌 Connecting to PostgreSQL...")
    conn = psycopg2.connect(POSTGRES_CONNECTION)
    conn.autocommit = False
    print(f"✅ Connected")

    # Create table, insert, build index
    create_table(conn)

    print(f"\n⏱️  Timing insert...")
    t_insert_start = time.time()
    insert_corpus(conn, corpus_emb, corpus_ids)
    insert_time = time.time() - t_insert_start
    print(f"   Insert time: {insert_time:.2f}s")

    build_time = build_index(conn, index_type="NONE")

    # Search with SQL
    results, total_search_time, avg_query_time = search_queries(
        conn, query_emb, query_ids, quantizer_params, top_k=10
    )

    # Evaluate
    metrics = evaluate(qrels, results)

    # Display
    print(f"\n{'='*60}")
    print("RESULTS - PostgreSQL SQ8 (Column-per-Dimension)")
    print(f"{'='*60}")
    print(f"\n📊 Accuracy Metrics:")
    print(f"   NDCG@1:      {metrics['ndcg@1']:.4f}")
    print(f"   NDCG@5:      {metrics['ndcg@5']:.4f}")
    print(f"   NDCG@10:     {metrics['ndcg@10']:.4f}")
    print(f"   Recall@1:    {metrics['recall@1']:.4f}")
    print(f"   Recall@5:    {metrics['recall@5']:.4f}")
    print(f"   Recall@10:   {metrics['recall@10']:.4f}")
    print(f"   Precision@1: {metrics['precision@1']:.4f}")
    print(f"   Precision@5: {metrics['precision@5']:.4f}")
    print(f"   Precision@10:{metrics['precision@10']:.4f}")
    print(f"   MAP@10:      {metrics['map@10']:.4f}")

    print(f"\n⏱️  Timing Metrics:")
    print(f"   Insert time:       {insert_time:.2f}s")
    print(f"   Index build time:  {build_time:.2f}s")
    print(f"   Total search time: {total_search_time:.2f}s")
    print(f"   Avg per query:     {avg_query_time*1000:.2f}ms")

    # Save results
    output = {
        "database":               "PostgreSQL",
        "method":                 "SQ8_column_per_dimension",
        "quantization":           "SQ8",
        "schema":                 f"{DIM} integer columns (x1…x{DIM})",
        "retrieval":              "SQL dot product: CAST(x AS INT) * q_float",
        "insert_time_sec":        round(insert_time, 2),
        "index_build_time_sec":   round(build_time, 2),
        "total_search_time_sec":  round(total_search_time, 2),
        "avg_query_time_ms":      round(avg_query_time * 1000, 2),
        **{k: round(v, 4) for k, v in metrics.items()}
    }

    os.makedirs("./results", exist_ok=True)
    with open("./results/postgres_sq8_column_results_k10.json", 'w') as f:
        json.dump(output, f, indent=2)

    print(f"\n💾 Results saved to: ./results/postgres_sq8_column_results_k10.json")

    conn.close()
    print(f"\n🎉 PostgreSQL SQ8 benchmark complete!")

    return output

if __name__ == "__main__":
    main()

POSTGRESQL SQ8 BENCHMARK ('s Column-per-Dimension Approach)

📐 Schema: 1 row per document, 1024 integer columns (x1…x1024)
🔍 Retrieval: Pure SQL dot product - no pgvector needed!

📥 Loading SQ8 quantized embeddings...
✅ Corpus: (13867, 1024), dtype=uint8
✅ Queries: (216, 1024), dtype=uint8
   Quantizer scale=0.001143, offset=-0.124361
✅ Loaded 64 query judgments

🔌 Connecting to PostgreSQL...
✅ Connected

📦 Creating table: finder_sq8_columns
   Schema: id, doc_id, x1, x2, ..., x1024  (1026 columns total)
✅ Table created with 1026 columns

⏱️  Timing insert...

📤 Inserting 13867 documents (1024 columns each)...
   Inserted 2000/13867 documents


   Inserted 4000/13867 documents
   Inserted 6000/13867 documents
   Inserted 8000/13867 documents
   Inserted 10000/13867 documents
   Inserted 12000/13867 documents
   Inserted 13867/13867 documents
✅ All 13867 documents inserted
   Insert time: 11.04s

🔨 Index setup: NONE
   No vector index - pure sequential SQL scan
✅ Index setup complete in 0.00s

🔍 Searching 216 queries using 's SQL...
   Method: Pure SQL dot product (CAST(x AS INT) * q_float)
   Table: finder_sq8_columns (1024 integer columns)
   Processed 20/216 queries (avg: 336.2ms/query)
   Processed 40/216 queries (avg: 343.7ms/query)
   Processed 60/216 queries (avg: 350.2ms/query)
   Processed 80/216 queries (avg: 359.0ms/query)
   Processed 100/216 queries (avg: 364.7ms/query)
   Processed 120/216 queries (avg: 366.0ms/query)
   Processed 140/216 queries (avg: 375.9ms/query)
   Processed 160/216 queries (avg: 380.4ms/query)
   Processed 180/216 queries (avg: 413.9ms/query)
   Processed 200/216 queries (avg: 396.0ms/query